In [1]:
import numpy as np
import pandas as pd
import json 
import os

from tqdm import tqdm

In [8]:
with open("parallel_sentences_cleaned.json","r",encoding="utf-8") as f:
    data = json.load(f)

print(data[0])


{'kannada': 'ಹುĪ', 'transliteration': 'huḷi', 'english': 'sour', 'page': 31, 'block_start': 6, 'block_end': 6}


In [9]:
eng_1word = {}
kannada_1word = {}
other_sentences = []

for item in data:
    eng_words = item['english'].split()
    kannada_words = item['transliteration'].split()
    
    # Check if English is 1 word
    if len(eng_words) == 1:
        eng_1word[item['transliteration']] = item['english']
    # Check if Kannada is 1 word but English is not
    elif len(kannada_words) == 1 and len(eng_words) > 1:
        kannada_1word[item['transliteration']] = item['english']
    # Everything else
    else:
        other_sentences.append({
            'transliteration': item['transliteration'],
            'english': item['english']
        })

print(f"Number of 1-word English sentences: {len(eng_1word)}")
print(f"Number of 1-word Kannada sentences: {len(kannada_1word)}")
print(f"Number of other sentences: {len(other_sentences)}")


Number of 1-word English sentences: 413
Number of 1-word Kannada sentences: 396
Number of other sentences: 393


In [10]:
# Display the first 10 sentences of 1 word english
# Display first 10 items from eng_1word dictionary
for i, (kannada, english) in enumerate(list(eng_1word.items())[:10]):
    print(f"{i+1}. Kannada: {kannada} -> English: {english}")


1. Kannada: huḷi -> English: sour
2. Kannada: huli -> English: tiger
3. Kannada: haḷḷi -> English: village
4. Kannada: halli -> English: lizard
5. Kannada: maṇe -> English: stool
6. Kannada: mane -> English: house
7. Kannada: hālu -> English: milk
8. Kannada: hallu -> English: tooth
9. Kannada: dēva -> English: god
10. Kannada: devva -> English: spirit


In [11]:
# Display first 10 items from kannada_1word dictionary
for i, (kannada, english) in enumerate(list(kannada_1word.items())[:10]):
    print(f"{i+1}. Kannada: {kannada} -> English: {english}")


1. Kannada: ōḍu -> English: to run
2. Kannada: ōdu -> English: to read
3. Kannada: hēḷu -> English: to say
4. Kannada: hēlu -> English: to shit
5. Kannada: baḍi -> English: to beat
6. Kannada: baḍḍi -> English: interest (on money)
7. Kannada: maḍi -> English: ritually pure
8. Kannada: māḍi -> English: please do The two following pairs of words are prosodically similar (a long sylla- ble followed by a short one), but whereas in the first word of both pairs the long syllable is long because the vowel is long, in the second word it is because of the doubled consonant, and this difference is heard in pronunciation:
9. Kannada: nīnu -> English: you (singular and non-honorific)
10. Kannada: nīvu -> English: you (plural / honorific)


In [12]:
# POS Tagging
import spacy
nlp = spacy.load("en_core_web_sm")

def pos_tagging(sentences):
    pos_tags = []
    for sentence in tqdm(sentences):
        doc = nlp(sentence)
        tags = [(token.text, token.pos_) for token in doc]
        pos_tags.append(tags)
    return pos_tags




In [13]:
# POS tagging for English words in eng_1word
eng_1word_sentences = list(eng_1word.values())
eng_1word_transliterations = list(eng_1word.keys())
eng_pos_tags = pos_tagging(eng_1word_sentences)




100%|██████████| 413/413 [00:01<00:00, 287.64it/s]


In [14]:
# Show all POS tags for the first 10 sentences
for i, tags in enumerate(eng_pos_tags[:100]):
    for word, pos in tags:
        print(f"{word} ({eng_1word_transliterations[i]}): {pos}")


sour (huḷi): ADJ
tiger (huli): NOUN
village (haḷḷi): NOUN
lizard (halli): NOUN
stool (maṇe): PROPN
house (mane): NOUN
milk (hālu): NOUN
tooth (hallu): VERB
god (dēva): PROPN
spirit (devva): PROPN
I (nānu): PRON
we (nāvu): PRON
he (avanu): PRON
she (avaḷu): PRON
he (ivanu): PRON
she (ivaḷu): PRON
here (illi): ADV
there (alli): ADV
where (elli): SCONJ
? (elli): PUNCT
now (īga): ADV
then (āga): ADV
that (adu): PRON
closet (almāri): NOUN
sky (ākāśa): NOUN
this (idu): PRON
another (innoṃdu): PRON
today (ivattu): NOUN
meal (ūṭa): NOUN
town (ūru): NOUN
two (eraḍu): NUM
five (aidu): NUM
one (oṃdu): NUM
computer (kaṃpyūṭaru): NOUN
building (kaṭṭaḍa): VERB
paper (kāgada): NOUN
forest (kāḍu): NOUN
car (kāru): NOUN
window (kiṭaki): NOUN
chair (kurci): NOUN
glass (gilāsu): NOUN
goal (guri): NOUN
teacher (guru): NOUN
Thursday (guruvāra): PROPN
spoon (camaca): VERB
knife (cāku): NOUN
mat (cāpe): NOUN
key (cāvi): ADJ
picture (citra): NOUN
taxi (ṭyāksi): NOUN
tubelight (ṭyūbulaiṭu): NOUN
plate (taṭṭe):

In [15]:
# map the transliterations to the POS tags 
pos_mapping = {}
for i, tags in enumerate(eng_pos_tags):
    transliteration = eng_1word_transliterations[i]
    pos_mapping[transliteration] = [pos for word, pos in tags]

pos_mapping

{'huḷi': ['ADJ'],
 'huli': ['NOUN'],
 'haḷḷi': ['NOUN'],
 'halli': ['NOUN'],
 'maṇe': ['PROPN'],
 'mane': ['NOUN'],
 'hālu': ['NOUN'],
 'hallu': ['VERB'],
 'dēva': ['PROPN'],
 'devva': ['PROPN'],
 'nānu': ['PRON'],
 'nāvu': ['PRON'],
 'avanu': ['PRON'],
 'avaḷu': ['PRON'],
 'ivanu': ['PRON'],
 'ivaḷu': ['PRON'],
 'illi': ['ADV'],
 'alli': ['ADV'],
 'elli': ['SCONJ', 'PUNCT'],
 'īga': ['ADV'],
 'āga': ['ADV'],
 'adu': ['PRON'],
 'almāri': ['NOUN'],
 'ākāśa': ['NOUN'],
 'idu': ['PRON'],
 'innoṃdu': ['PRON'],
 'ivattu': ['NOUN'],
 'ūṭa': ['NOUN'],
 'ūru': ['NOUN'],
 'eraḍu': ['NUM'],
 'aidu': ['NUM'],
 'oṃdu': ['NUM'],
 'kaṃpyūṭaru': ['NOUN'],
 'kaṭṭaḍa': ['VERB'],
 'kāgada': ['NOUN'],
 'kāḍu': ['NOUN'],
 'kāru': ['NOUN'],
 'kiṭaki': ['NOUN'],
 'kurci': ['NOUN'],
 'gilāsu': ['NOUN'],
 'guri': ['NOUN'],
 'guru': ['NOUN'],
 'guruvāra': ['PROPN'],
 'camaca': ['VERB'],
 'cāku': ['NOUN'],
 'cāpe': ['NOUN'],
 'cāvi': ['ADJ'],
 'citra': ['NOUN'],
 'ṭyāksi': ['NOUN'],
 'ṭyūbulaiṭu': ['NOUN'],
 't

In [16]:
kan_to_eng_mapping = {}
for transliteration, english in kannada_1word.items():
    kan_to_eng_mapping[transliteration] = english

for transliteration, english in eng_1word.items():
    if transliteration not in kan_to_eng_mapping:
        kan_to_eng_mapping[transliteration] = english

kan_to_eng_mapping

{'ōḍu': 'to run',
 'ōdu': 'to read',
 'hēḷu': 'to say',
 'hēlu': 'to shit',
 'baḍi': 'to beat',
 'baḍḍi': 'interest (on money)',
 'maḍi': 'ritually pure',
 'māḍi': 'please do The two following pairs of words are prosodically similar (a long sylla- ble followed by a short one), but whereas in the first word of both pairs the long syllable is long because the vowel is long, in the second word it is because of the doubled consonant, and this difference is heard in pronunciation:',
 'nīnu': 'you (singular and non-honorific)',
 'nīvu': 'you (plural / honorific)',
 'adu': 'it / that',
 'avaru': 'they (persons)',
 'avu': 'they (non-persons)',
 'idu': 'it / this',
 'ivaru': 'he / she (honorific)',
 'ivu': 'they / these (neuter) The proximate pronouns are used when the objects to which they refer are near to the speaker. (Whenever the distantness or proximity of the object is unclear, irrelevant or unimportant, the distant pronouns are generally used.)',
 'hīge': 'in this manner',
 'hāge': 'in 

In [17]:
import json

# Save the kannada to english mapping to a json file called kan_to_eng_dict.json
with open("kan_to_eng_dict.json", "w", encoding="utf-8") as f:
    json.dump(kan_to_eng_mapping, f, ensure_ascii=False, indent=4)
# Save the POS mapping to a json file called pos_mapping.json
with open("pos_mapping.json", "w", encoding="utf-8") as f:
    json.dump(pos_mapping, f, ensure_ascii=False, indent=4)

    

In [18]:
list(kannada_1word.values())

['to run',
 'to read',
 'to say',
 'to shit',
 'to beat',
 'interest (on money)',
 'ritually pure',
 'please do The two following pairs of words are prosodically similar (a long sylla- ble followed by a short one), but whereas in the first word of both pairs the long syllable is long because the vowel is long, in the second word it is because of the doubled consonant, and this difference is heard in pronunciation:',
 'you (singular and non-honorific)',
 'you (plural / honorific)',
 'it / that',
 'they (persons)',
 'they (non-persons)',
 'it / this',
 'he / she (honorific)',
 'they / these (neuter) The proximate pronouns are used when the objects to which they refer are near to the speaker. (Whenever the distantness or proximity of the object is unclear, irrelevant or unimportant, the distant pronouns are generally used.)',
 'in this manner',
 'in that manner',
 'in which manner?',
 'this much',
 'that much',
 'how much?',
 'it is, it exists',
 'mouse, rat',
 'leg, foot',
 'hand, arm',


In [19]:
# POS tag for Kannada words in kannada_1word

kannada_1word_sentences = list(kannada_1word.values())
# For each sentence, remove anything inside parentheses
kannada_1word_sentences = [sentence.split('(')[0].strip() for sentence in kannada_1word_sentences]

# TODO: More preprocessing 
# 1) / implies can mean two words, so either we can split it
# 2) ? at the end of a sentence implies a question, so we can add question

kannada_1word_sentences





['to run',
 'to read',
 'to say',
 'to shit',
 'to beat',
 'interest',
 'ritually pure',
 'please do The two following pairs of words are prosodically similar',
 'you',
 'you',
 'it / that',
 'they',
 'they',
 'it / this',
 'he / she',
 'they / these',
 'in this manner',
 'in that manner',
 'in which manner?',
 'this much',
 'that much',
 'how much?',
 'it is, it exists',
 'mouse, rat',
 'leg, foot',
 'hand, arm',
 'book, volume',
 'big, large',
 '',
 'ballpoint pen',
 'pair of trousers',
 'hill, mountain',
 'old woman',
 'lawn, field',
 'young woman',
 '',
 '[male] student',
 '[female] student',
 '',
 '',
 'cow dung',
 'things, belongings',
 '',
 'singer',
 'singer',
 '',
 'Yes, you may go',
 'How are you? Gururāja',
 'his / her',
 'from where?',
 'nicely, finely',
 'your',
 'please come',
 'I should come',
 'to come',
 'bus',
 'is wanted / required / needed',
 'is not wanted / not required',
 'something special',
 'okay, all right, correct',
 'elder sister',
 'elder brother',
 'tank,

In [20]:
kannada_1word_transliterations = list(kannada_1word.keys())
kannada_pos_tags = pos_tagging(kannada_1word_sentences)
# Show all POS tags for the first 10 sentences

100%|██████████| 396/396 [00:01<00:00, 291.18it/s]


In [21]:
kannada_pos_tags

[[('to', 'PART'), ('run', 'VERB')],
 [('to', 'PART'), ('read', 'VERB')],
 [('to', 'PART'), ('say', 'VERB')],
 [('to', 'PART'), ('shit', 'VERB')],
 [('to', 'PART'), ('beat', 'VERB')],
 [('interest', 'NOUN')],
 [('ritually', 'ADV'), ('pure', 'ADJ')],
 [('please', 'INTJ'),
  ('do', 'VERB'),
  ('The', 'DET'),
  ('two', 'NUM'),
  ('following', 'VERB'),
  ('pairs', 'NOUN'),
  ('of', 'ADP'),
  ('words', 'NOUN'),
  ('are', 'AUX'),
  ('prosodically', 'ADV'),
  ('similar', 'ADJ')],
 [('you', 'PRON')],
 [('you', 'PRON')],
 [('it', 'PRON'), ('/', 'PUNCT'), ('that', 'PRON')],
 [('they', 'PRON')],
 [('they', 'PRON')],
 [('it', 'PRON'), ('/', 'PUNCT'), ('this', 'PRON')],
 [('he', 'PRON'), ('/', 'SYM'), ('she', 'PRON')],
 [('they', 'PRON'), ('/', 'PUNCT'), ('these', 'PRON')],
 [('in', 'ADP'), ('this', 'DET'), ('manner', 'NOUN')],
 [('in', 'ADP'), ('that', 'DET'), ('manner', 'NOUN')],
 [('in', 'ADP'), ('which', 'PRON'), ('manner', 'NOUN'), ('?', 'PUNCT')],
 [('this', 'PRON'), ('much', 'ADJ')],
 [('that

In [22]:
# Show all POS tags for the first 10 sentences
for i, tags in enumerate(kannada_pos_tags[:100]):
    print(tags, f" - ({kannada_1word_transliterations[i]})")  

[('to', 'PART'), ('run', 'VERB')]  - (ōḍu)
[('to', 'PART'), ('read', 'VERB')]  - (ōdu)
[('to', 'PART'), ('say', 'VERB')]  - (hēḷu)
[('to', 'PART'), ('shit', 'VERB')]  - (hēlu)
[('to', 'PART'), ('beat', 'VERB')]  - (baḍi)
[('interest', 'NOUN')]  - (baḍḍi)
[('ritually', 'ADV'), ('pure', 'ADJ')]  - (maḍi)
[('please', 'INTJ'), ('do', 'VERB'), ('The', 'DET'), ('two', 'NUM'), ('following', 'VERB'), ('pairs', 'NOUN'), ('of', 'ADP'), ('words', 'NOUN'), ('are', 'AUX'), ('prosodically', 'ADV'), ('similar', 'ADJ')]  - (māḍi)
[('you', 'PRON')]  - (nīnu)
[('you', 'PRON')]  - (nīvu)
[('it', 'PRON'), ('/', 'PUNCT'), ('that', 'PRON')]  - (adu)
[('they', 'PRON')]  - (avaru)
[('they', 'PRON')]  - (avu)
[('it', 'PRON'), ('/', 'PUNCT'), ('this', 'PRON')]  - (idu)
[('he', 'PRON'), ('/', 'SYM'), ('she', 'PRON')]  - (ivaru)
[('they', 'PRON'), ('/', 'PUNCT'), ('these', 'PRON')]  - (ivu)
[('in', 'ADP'), ('this', 'DET'), ('manner', 'NOUN')]  - (hīge)
[('in', 'ADP'), ('that', 'DET'), ('manner', 'NOUN')]  - (hāge

In [23]:
# Map the transliterations to the POS tags
kannada_pos_mapping = {}
for i, tags in enumerate(kannada_pos_tags):
    transliteration = kannada_1word_transliterations[i]
    kannada_pos_mapping[transliteration] = [pos for word, pos in tags]  

kannada_pos_mapping

{'ōḍu': ['PART', 'VERB'],
 'ōdu': ['PART', 'VERB'],
 'hēḷu': ['PART', 'VERB'],
 'hēlu': ['PART', 'VERB'],
 'baḍi': ['PART', 'VERB'],
 'baḍḍi': ['NOUN'],
 'maḍi': ['ADV', 'ADJ'],
 'māḍi': ['INTJ',
  'VERB',
  'DET',
  'NUM',
  'VERB',
  'NOUN',
  'ADP',
  'NOUN',
  'AUX',
  'ADV',
  'ADJ'],
 'nīnu': ['PRON'],
 'nīvu': ['PRON'],
 'adu': ['PRON', 'PUNCT', 'PRON'],
 'avaru': ['PRON'],
 'avu': ['PRON'],
 'idu': ['PRON', 'PUNCT', 'PRON'],
 'ivaru': ['PRON', 'SYM', 'PRON'],
 'ivu': ['PRON', 'PUNCT', 'PRON'],
 'hīge': ['ADP', 'DET', 'NOUN'],
 'hāge': ['ADP', 'DET', 'NOUN'],
 'hēge': ['ADP', 'PRON', 'NOUN', 'PUNCT'],
 'iṣṭu': ['PRON', 'ADJ'],
 'aṣṭu': ['ADV', 'ADV'],
 'eṣṭu': ['SCONJ', 'ADJ', 'PUNCT'],
 'ide': ['PRON', 'AUX', 'PUNCT', 'PRON', 'VERB'],
 'ili': ['PROPN', 'PUNCT', 'CCONJ'],
 'kālu': ['PROPN', 'PUNCT', 'NOUN'],
 'kai': ['NOUN', 'PUNCT', 'VERB'],
 'graṃtha': ['NOUN', 'PUNCT', 'NOUN'],
 'doḍḍa': ['ADJ', 'PUNCT', 'ADJ'],
 'paṃce': [],
 'pennu': ['PROPN', 'NOUN'],
 'pyāṃṭu': ['NOUN', '

In [24]:
# POS Tagging for all other sentences
other_sentences_sentences = [item['english'] for item in other_sentences]

other_sentences_transliterations = [item['transliteration'] for item in other_sentences]
other_pos_tags = pos_tagging(other_sentences_sentences)


100%|██████████| 393/393 [00:01<00:00, 202.42it/s]


In [25]:
# Display first 10 items from other pos tags
for i, tags in enumerate(other_pos_tags[:10]):
    print(f"{i+1}. Transliteration: {other_sentences_transliterations[i]}")
    for word, pos in tags:
        print(f"   {word}: {pos}")
    
    transliteration = other_sentences_transliterations[i].lower()
    transliteration = transliteration.strip()
    # Remove any trailing punctuation
    transliteration = transliteration.rstrip('.,!?')
    # Print the transliteration

    transliteration = transliteration.split()
    # Check if the transliteration is in the pos_mapping
    for word in transliteration:
        word = word.lower()
        if word in pos_mapping:
            print(f"POS tags for {word}: {pos_mapping[word]}")
        elif word in kannada_pos_mapping:
            print(f"POS tags for {word}: {kannada_pos_mapping[word]}")
        else:
            print(f"No POS tags found for {word}")
    print("=========================================")

    


1. Transliteration: Idu pustaka.
   This: PRON
   is: AUX
   a: DET
   book: NOUN
   .: PUNCT
POS tags for idu: ['PRON']
POS tags for pustaka: ['NOUN']
2. Transliteration: Ivu pustakagaḷu.
   These: PRON
   are: AUX
   books: NOUN
   .: PUNCT
POS tags for ivu: ['PRON', 'PUNCT', 'PRON']
No POS tags found for pustakagaḷu
3. Transliteration: Adu mara.
   That: PRON
   is: AUX
   a: DET
   tree: NOUN
   .: PUNCT
POS tags for adu: ['PRON']
POS tags for mara: ['NOUN']
4. Transliteration: Avu maragaḷu.
   Those: PRON
   are: AUX
   trees: NOUN
   .: PUNCT
POS tags for avu: ['PRON']
No POS tags found for maragaḷu
5. Transliteration: Adu huḍuga.
   That: PRON
   is: AUX
   a: DET
   boy: NOUN
   .: PUNCT
POS tags for adu: ['PRON']
POS tags for huḍuga: ['PROPN']
6. Transliteration: Adu mane.
   That: PRON
   is: AUX
   a: DET
   house: NOUN
   .: PUNCT
POS tags for adu: ['PRON']
POS tags for mane: ['NOUN']
7. Transliteration: Avu manegaḷu.
   Those: PRON
   are: AUX
   houses: NOUN
   .: PUNCT
P

In [26]:
all_pos_mapping = {}
# Combine all mappings  
all_pos_mapping.update(pos_mapping)
all_pos_mapping.update(kannada_pos_mapping)

all_pos_mapping

{'huḷi': ['ADJ'],
 'huli': ['NOUN'],
 'haḷḷi': ['NOUN'],
 'halli': ['NOUN'],
 'maṇe': ['PROPN'],
 'mane': ['NOUN'],
 'hālu': ['NOUN'],
 'hallu': ['VERB'],
 'dēva': ['PROPN'],
 'devva': ['PROPN'],
 'nānu': ['PRON'],
 'nāvu': ['PRON'],
 'avanu': ['PRON'],
 'avaḷu': ['PRON'],
 'ivanu': ['PRON'],
 'ivaḷu': ['PRON'],
 'illi': ['ADV'],
 'alli': ['ADV'],
 'elli': ['SCONJ', 'PUNCT'],
 'īga': ['ADV'],
 'āga': ['ADV', 'PUNCT', 'DET', 'NOUN'],
 'adu': ['PRON', 'PUNCT', 'PRON'],
 'almāri': ['NOUN'],
 'ākāśa': ['NOUN'],
 'idu': ['PRON', 'PUNCT', 'PRON'],
 'innoṃdu': ['PRON'],
 'ivattu': ['NOUN'],
 'ūṭa': [],
 'ūru': ['NOUN',
  'PUNCT',
  'NOUN',
  'PUNCT',
  'NOUN',
  'PUNCT',
  'PUNCT',
  'ADJ',
  'NOUN',
  'PUNCT',
  'PUNCT',
  'NOUN'],
 'eraḍu': ['NUM'],
 'aidu': ['NUM'],
 'oṃdu': ['NUM', 'PUNCT', 'PRON'],
 'kaṃpyūṭaru': ['NOUN'],
 'kaṭṭaḍa': ['VERB'],
 'kāgada': ['NOUN'],
 'kāḍu': ['NOUN'],
 'kāru': ['NOUN'],
 'kiṭaki': ['NOUN'],
 'kurci': ['NOUN'],
 'gilāsu': ['NOUN'],
 'guri': ['NOUN'],
 'gur

In [27]:
# Display the dictionary
kan_to_eng_mapping

{'ōḍu': 'to run',
 'ōdu': 'to read',
 'hēḷu': 'to say',
 'hēlu': 'to shit',
 'baḍi': 'to beat',
 'baḍḍi': 'interest (on money)',
 'maḍi': 'ritually pure',
 'māḍi': 'please do The two following pairs of words are prosodically similar (a long sylla- ble followed by a short one), but whereas in the first word of both pairs the long syllable is long because the vowel is long, in the second word it is because of the doubled consonant, and this difference is heard in pronunciation:',
 'nīnu': 'you (singular and non-honorific)',
 'nīvu': 'you (plural / honorific)',
 'adu': 'it / that',
 'avaru': 'they (persons)',
 'avu': 'they (non-persons)',
 'idu': 'it / this',
 'ivaru': 'he / she (honorific)',
 'ivu': 'they / these (neuter) The proximate pronouns are used when the objects to which they refer are near to the speaker. (Whenever the distantness or proximity of the object is unclear, irrelevant or unimportant, the distant pronouns are generally used.)',
 'hīge': 'in this manner',
 'hāge': 'in 

In [28]:
# Create a dict with the values as dict of transation and pos tag
all_pos_mapping_dict = {}
for transliteration, pos_tags in all_pos_mapping.items():
    all_pos_mapping_dict[transliteration] = {
        'translation': kan_to_eng_mapping.get(transliteration, ""),
        "pos_tags": pos_tags
    }

# Save
with open("all_pos_mapping_with_translation.json", "w", encoding="utf-8") as f:
    json.dump(all_pos_mapping_dict, f, ensure_ascii=False, indent=4)

all_pos_mapping_dict



{'huḷi': {'translation': 'sour', 'pos_tags': ['ADJ']},
 'huli': {'translation': 'tiger', 'pos_tags': ['NOUN']},
 'haḷḷi': {'translation': 'village', 'pos_tags': ['NOUN']},
 'halli': {'translation': 'lizard', 'pos_tags': ['NOUN']},
 'maṇe': {'translation': 'stool', 'pos_tags': ['PROPN']},
 'mane': {'translation': 'house', 'pos_tags': ['NOUN']},
 'hālu': {'translation': 'milk', 'pos_tags': ['NOUN']},
 'hallu': {'translation': 'tooth', 'pos_tags': ['VERB']},
 'dēva': {'translation': 'god', 'pos_tags': ['PROPN']},
 'devva': {'translation': 'spirit', 'pos_tags': ['PROPN']},
 'nānu': {'translation': 'I', 'pos_tags': ['PRON']},
 'nāvu': {'translation': 'we', 'pos_tags': ['PRON']},
 'avanu': {'translation': 'he', 'pos_tags': ['PRON']},
 'avaḷu': {'translation': 'she', 'pos_tags': ['PRON']},
 'ivanu': {'translation': 'he', 'pos_tags': ['PRON']},
 'ivaḷu': {'translation': 'she', 'pos_tags': ['PRON']},
 'illi': {'translation': 'here', 'pos_tags': ['ADV']},
 'alli': {'translation': 'there', 'pos_t

In [29]:
all_pos_mapping

{'huḷi': ['ADJ'],
 'huli': ['NOUN'],
 'haḷḷi': ['NOUN'],
 'halli': ['NOUN'],
 'maṇe': ['PROPN'],
 'mane': ['NOUN'],
 'hālu': ['NOUN'],
 'hallu': ['VERB'],
 'dēva': ['PROPN'],
 'devva': ['PROPN'],
 'nānu': ['PRON'],
 'nāvu': ['PRON'],
 'avanu': ['PRON'],
 'avaḷu': ['PRON'],
 'ivanu': ['PRON'],
 'ivaḷu': ['PRON'],
 'illi': ['ADV'],
 'alli': ['ADV'],
 'elli': ['SCONJ', 'PUNCT'],
 'īga': ['ADV'],
 'āga': ['ADV', 'PUNCT', 'DET', 'NOUN'],
 'adu': ['PRON', 'PUNCT', 'PRON'],
 'almāri': ['NOUN'],
 'ākāśa': ['NOUN'],
 'idu': ['PRON', 'PUNCT', 'PRON'],
 'innoṃdu': ['PRON'],
 'ivattu': ['NOUN'],
 'ūṭa': [],
 'ūru': ['NOUN',
  'PUNCT',
  'NOUN',
  'PUNCT',
  'NOUN',
  'PUNCT',
  'PUNCT',
  'ADJ',
  'NOUN',
  'PUNCT',
  'PUNCT',
  'NOUN'],
 'eraḍu': ['NUM'],
 'aidu': ['NUM'],
 'oṃdu': ['NUM', 'PUNCT', 'PRON'],
 'kaṃpyūṭaru': ['NOUN'],
 'kaṭṭaḍa': ['VERB'],
 'kāgada': ['NOUN'],
 'kāḍu': ['NOUN'],
 'kāru': ['NOUN'],
 'kiṭaki': ['NOUN'],
 'kurci': ['NOUN'],
 'gilāsu': ['NOUN'],
 'guri': ['NOUN'],
 'gur

# Quick affix separation test

In [30]:
processed_sentences = []

desc_sorted_all_pos_mapping = dict(sorted(all_pos_mapping.items(), key=lambda item: len(item[0]), reverse=True))
for i, sentence in enumerate(other_sentences_transliterations):

    translated_sentence = other_sentences_sentences[i]
    transliteration = sentence.lower()
    transliteration = transliteration.strip()
    # Remove any trailing punctuation
    
    # Split into words
    transliteration = transliteration.split()
    # Remove trailing punctuation from each word
    transliteration = [word.rstrip('.,!?') for word in transliteration]
    #print(transliteration)
    result_dict = {}
    
    # Check if the transliteration is in the pos_mapping
    for word in transliteration:
        word = word.lower()
        if word in all_pos_mapping:
            # Check if the word is in kan_to_eng_mapping
            if word in kan_to_eng_mapping:
                translation = kan_to_eng_mapping[word]
            else:
                print(f"No translation found for {word}")
                translation = None
            result_dict[word] = {"pos": all_pos_mapping[word], "translation": translation, "affix": None}
        else:
            #print(f"No POS tags found for {word}")
            # Find if the word is affixed form of any word in all_pos_mapping
            found = False
            for key in desc_sorted_all_pos_mapping.keys():
                temp_key = key
                key = key.lower()
                if word.find(key) != -1:
                    found = True
                    #print(f"Found affixed form of {key} in {word}")
                    #print(transliteration)
                    affix = word.replace(key, "______")
                    prefix, suffix = affix.split("______")
                    #print(f"Prefix: {prefix} Suffix: {suffix}")
                    result_dict[word] = {
                        "pos": all_pos_mapping[temp_key],
                        "translation": kan_to_eng_mapping.get(temp_key, None),
                        "affix": {
                            "prefix": prefix,
                            "suffix": suffix
                        }
                    }
                    
                    break
            if not found:
                print(f"No affixed form found for '{word}' do more fuzzy matching (TODO)")
                result_dict[word] = {"pos": None, "translation": None, "affix": None}
    processed_sentences.append((sentence, translated_sentence, result_dict))
    #print("=========================================")


No affixed form found for 'adū' do more fuzzy matching (TODO)
No affixed form found for 'alla' do more fuzzy matching (TODO)
No affixed form found for 'toḍak-alla' do more fuzzy matching (TODO)
No affixed form found for 'toḍak-illa' do more fuzzy matching (TODO)
No affixed form found for 'adū' do more fuzzy matching (TODO)
No affixed form found for 'adū' do more fuzzy matching (TODO)
No affixed form found for 'adū' do more fuzzy matching (TODO)
No affixed form found for 'illa' do more fuzzy matching (TODO)
No affixed form found for 'illa' do more fuzzy matching (TODO)
No affixed form found for 'bekku' do more fuzzy matching (TODO)
No affixed form found for 'dhāravāḍa' do more fuzzy matching (TODO)
No affixed form found for 'karnāṭakada' do more fuzzy matching (TODO)
No affixed form found for 'ēnū' do more fuzzy matching (TODO)
No affixed form found for 'illa' do more fuzzy matching (TODO)
No affixed form found for 'nīvū' do more fuzzy matching (TODO)
No affixed form found for 'illa' do

In [31]:
for result in processed_sentences:
    sentence, translated_sentence,pos_info = result
    print(f"Sentence: {sentence}")
    print(f"Translated Sentence: {translated_sentence}")
    for word, info in pos_info.items():
        print(f"  Word: {word}, POS: {info['pos']}, Translation: {info['translation']}, Affix: {info['affix']}")
    print("=========================================")

Sentence: Idu pustaka.
Translated Sentence: This is a book.
  Word: idu, POS: ['PRON', 'PUNCT', 'PRON'], Translation: it / this, Affix: None
  Word: pustaka, POS: ['NOUN'], Translation: book, Affix: None
Sentence: Ivu pustakagaḷu.
Translated Sentence: These are books.
  Word: ivu, POS: ['PRON', 'PUNCT', 'PRON'], Translation: they / these (neuter) The proximate pronouns are used when the objects to which they refer are near to the speaker. (Whenever the distantness or proximity of the object is unclear, irrelevant or unimportant, the distant pronouns are generally used.), Affix: None
  Word: pustakagaḷu, POS: ['NOUN'], Translation: book, Affix: {'prefix': '', 'suffix': 'gaḷu'}
Sentence: Adu mara.
Translated Sentence: That is a tree.
  Word: adu, POS: ['PRON', 'PUNCT', 'PRON'], Translation: it / that, Affix: None
  Word: mara, POS: ['NOUN', 'PUNCT', 'NOUN'], Translation: tree, wood, Affix: None
Sentence: Avu maragaḷu.
Translated Sentence: Those are trees.
  Word: avu, POS: ['PRON'], Tran

In [32]:
# Further splitting of prefix and suffix TODO:
for result in results:
    sentence, translated_sentence, pos_info = result
    for word, info in pos_info.items():
        if info['affix'] is not None:
            prefix = info['affix']['prefix']
            suffix = info['affix']['suffix']
            
            # Find suffxies
        

NameError: name 'results' is not defined

# Get list of common affixes and their frequencies, along with the different words that contain them

In [ ]:
from collections import defaultdict

# Words and suffixes detected
words_with_suffixes = defaultdict(list)
words_with_prefixes = defaultdict(list)
for result in results:
    sentence, translated_sentence, pos_info = result
    for word, info in pos_info.items():
        if info['affix'] is not None:
            prefix = info['affix']['prefix']
            suffix = info['affix']['suffix']
            
            # Find prefixes
            if prefix:
                words_with_prefixes[prefix].append(word)
            
            # Find suffixes
            if suffix:
                words_with_suffixes[suffix].append(word)


words_with_suffixes = dict(words_with_suffixes)

#words_with_prefixes = dict(words_with_prefixes)

words_with_suffixes

# TODO: I KNOW KANNADA HAS SUFFIXES, BUT HOW DO YOU DETECT THAT IN THE LANGUAGE?



{'gaḷu': ['pustakagaḷu',
  'maragaḷu',
  'manegaḷu',
  'gurugaḷu',
  'koṃbugaḷu',
  'maragaḷu',
  'pustakagaḷu',
  'pustakagaḷu',
  'maragaḷu',
  'maragaḷu',
  'pustakagaḷu',
  'pustakagaḷu',
  'pustakagaḷu',
  'haṇṇugaḷu',
  'maragaḷu',
  'haṇṇugaḷu',
  'cīlagaḷu',
  'pustakagaḷu',
  'vidyārthigaḷu',
  'pustakagaḷu'],
 'ru': ['purōhitaru',
  'raitaru',
  'raitaru',
  'huḍugaru',
  'huḍugaru',
  'tammaṃdiru',
  'janaru',
  'janaru'],
 'yā': ['vidyārthiyā', 'vidyārthiniyā'],
 'vā': ['maravā', 'pustakavā', 'pustakavā', 'pustakavā', 'maravā'],
 'valla': ['pustakavalla', 'maravalla', 'iṣṭavalla'],
 '-v-alla': ['mara-v-alla'],
 '-y-alla': ['huḍugi-y-alla'],
 '-v-illa': ['mara-v-illa'],
 '-y-illa': ['huḍugi-y-illa'],
 'mogga': ['śivamogga'],
 'ddīrā': ['cennāgiddīrā'],
 ')': ['(nānu)', '(aṣṭu)', '(aṣṭu)'],
 'rājarē': ['gururājarē'],
 'navarē': ['rāmappanavarē',
  'sītammanavarē',
  'sītammanavarē',
  'rāmayyanavarē'],
 'ddēne': ['cennāgiddēne',
  'cennāgiddēne',
  'cennāgiddēne',
  'cennāgid

In [ ]:
suffix_counts = defaultdict(lambda: defaultdict(int))   # suffix → {stem: freq}
# For each suffix, the dictionary will contain a dict counting stems that use it
for suffix, words in words_with_suffixes.items():
    for word in words:
        # Extract the stem by removing the suffix
        stem = word[:-len(suffix)] if len(suffix) > 0 else word
        suffix_counts[suffix][stem] += 1

In [ ]:
suffix_counts = dict(suffix_counts)
# Display the suffix counts
for suffix, stems in suffix_counts.items():
    print(f"Suffix: {suffix}")
    for stem, freq in stems.items():
        print(f"  Stem: {stem}, Frequency: {freq}")
    print("=========================================")

Suffix: gaḷu
  Stem: pustaka, Frequency: 8
  Stem: mara, Frequency: 5
  Stem: mane, Frequency: 1
  Stem: guru, Frequency: 1
  Stem: koṃbu, Frequency: 1
  Stem: haṇṇu, Frequency: 2
  Stem: cīla, Frequency: 1
  Stem: vidyārthi, Frequency: 1
Suffix: ru
  Stem: purōhita, Frequency: 1
  Stem: raita, Frequency: 2
  Stem: huḍuga, Frequency: 2
  Stem: tammaṃdi, Frequency: 1
  Stem: jana, Frequency: 2
Suffix: yā
  Stem: vidyārthi, Frequency: 1
  Stem: vidyārthini, Frequency: 1
Suffix: vā
  Stem: mara, Frequency: 2
  Stem: pustaka, Frequency: 3
Suffix: valla
  Stem: pustaka, Frequency: 1
  Stem: mara, Frequency: 1
  Stem: iṣṭa, Frequency: 1
Suffix: -v-alla
  Stem: mara, Frequency: 1
Suffix: -y-alla
  Stem: huḍugi, Frequency: 1
Suffix: -v-illa
  Stem: mara, Frequency: 1
Suffix: -y-illa
  Stem: huḍugi, Frequency: 1
Suffix: mogga
  Stem: śiva, Frequency: 1
Suffix: ddīrā
  Stem: cennāgi, Frequency: 1
Suffix: )
  Stem: (nānu, Frequency: 1
  Stem: (aṣṭu, Frequency: 2
Suffix: rājarē
  Stem: guru, Frequ

In [ ]:
# Retain productive suffixes
productive_suffixes = {suffix: stems for suffix, stems in suffix_counts.items() if len(stems) > 1}

# Display productive suffixes
for suffix, stems in productive_suffixes.items():
    print(f"Productive Suffix: {suffix}")
    for stem, freq in stems.items():
        print(f"  Stem: {stem}, Frequency: {freq}")
    print("=========================================")
    

Productive Suffix: gaḷu
  Stem: pustaka, Frequency: 8
  Stem: mara, Frequency: 5
  Stem: mane, Frequency: 1
  Stem: guru, Frequency: 1
  Stem: koṃbu, Frequency: 1
  Stem: haṇṇu, Frequency: 2
  Stem: cīla, Frequency: 1
  Stem: vidyārthi, Frequency: 1
Productive Suffix: ru
  Stem: purōhita, Frequency: 1
  Stem: raita, Frequency: 2
  Stem: huḍuga, Frequency: 2
  Stem: tammaṃdi, Frequency: 1
  Stem: jana, Frequency: 2
Productive Suffix: yā
  Stem: vidyārthi, Frequency: 1
  Stem: vidyārthini, Frequency: 1
Productive Suffix: vā
  Stem: mara, Frequency: 2
  Stem: pustaka, Frequency: 3
Productive Suffix: valla
  Stem: pustaka, Frequency: 1
  Stem: mara, Frequency: 1
  Stem: iṣṭa, Frequency: 1
Productive Suffix: )
  Stem: (nānu, Frequency: 1
  Stem: (aṣṭu, Frequency: 2
Productive Suffix: navarē
  Stem: rāmappa, Frequency: 1
  Stem: sītamma, Frequency: 2
  Stem: rāmayya, Frequency: 1
Productive Suffix: ddēne
  Stem: cennāgi, Frequency: 4
  Stem: māḍi, Frequency: 1
  Stem: hōgi, Frequency: 1
Prod

In [ ]:
# Take all words in the sentences and get stems with suffixes
all_words = set()
for result in results:
    sentence, translated_sentence, pos_info = result
    for word in pos_info.keys():
        all_words.add(word) 
# Display all words
print(f"Total unique words in sentences: {len(all_words)}")


Total unique words in sentences: 593


In [ ]:
#all_words
# Remove all punctutation from the words
all_words = [word.strip('.,!?()[]/"!') for word in all_words]

all_words = set(all_words)  # Convert to set to remove duplicates

# Display all words
print(f"Total unique words in sentences after removing punctuation: {len(all_words)}")

Total unique words in sentences after removing punctuation: 590


In [ ]:
all_words

{'',
 '12',
 '2',
 '4',
 'adakkiṃta',
 'adannu',
 'adu',
 'adū',
 'aidaralli',
 'aidu',
 'alla',
 'alli',
 'amma',
 'anna',
 'aparūpavāgi',
 'araḷida',
 'araḷidudannu',
 'arthaisu',
 'arthavisu',
 'asādhya',
 'athavā',
 'avana',
 'avanige',
 'avaniṃda',
 'avanu',
 'avara',
 'avaradakke',
 'avarannu',
 'avarige',
 'avaru',
 'avarū',
 'avaḷannu',
 'avaḷige',
 'avaḷu',
 'avu',
 'aḷabēkō',
 'aṃte',
 'aṣṭu',
 'ba-',
 'bagege',
 'bahaḷa',
 'balliri',
 'banni',
 'barabahudu',
 'barabahudā',
 'baralilla',
 'baralā',
 'bare-',
 'bareda',
 'baredaru',
 'baredilla',
 'baredubiḍuttāre',
 'baredukoḍuttēne',
 'baredukoḷḷuttēne',
 'bareyalilla',
 'bareyalu',
 'bareyuttalilla',
 'bareyuva',
 'bareyuvudilla',
 'bareyuvudu',
 'baruttade',
 'baruttadeyā',
 'baruttiddenu',
 'baruttāne',
 'baruttānō',
 'baruttāre',
 'baruttēne',
 'baruttēne”',
 'baruvahāge',
 'baruvudilla',
 'baruvudillaveṃdu',
 'bassu',
 'baṃda',
 'baṃdare',
 'baṃdaru',
 'baṃdarū',
 'baṃdavanu',
 'baṃdaḷu',
 'baṃdiddēne',
 'baṃdāga',
 'ba

In [ ]:
# Find stems in all_words that have suffixes we found in productive_suffixes
stems_with_suffixes = defaultdict(list)  # stem → [suffix1, suffix2, ...]
for word in all_words:
    for suffix in productive_suffixes.keys():
        if word.endswith(suffix):
            stem = word[:-len(suffix)] if len(suffix) > 0 else word
            stems_with_suffixes[stem].append(suffix)
# Convert to a regular dictionary
stems_with_suffixes = dict(stems_with_suffixes)
# Display stems with their suffixes
for stem, suffixes in stems_with_suffixes.items():
    print(f"Stem: {stem}, Suffixes: {suffixes}")

Stem: para, Suffixes: ['vā']
Stem: baruvahā, Suffixes: ['ge']
Stem: idan, Suffixes: ['nu']
Stem: ida, Suffixes: ['nnu']
Stem: vidyārthi, Suffixes: ['gaḷu', 'yā']
Stem: i, Suffixes: ['du', 'lla', 'de', 'varu']
Stem: saṃkōcapaḍisu, Suffixes: ['ttāne']
Stem: samācāra, Suffixes: ['villa']
Stem: samācāravi, Suffixes: ['lla']
Stem: hēḷida, Suffixes: ['ru', 'nu']
Stem: hēḷi, Suffixes: ['daru', 'da', 'ddēne']
Stem: viṣaya, Suffixes: ['vannu']
Stem: viṣayavan, Suffixes: ['nu']
Stem: viṣayava, Suffixes: ['nnu']
Stem: māḍu, Suffixes: ['va', 'vudilla', 'ttēne', 'venu']
Stem: āgutta, Suffixes: ['de']
Stem: āgu, Suffixes: ['ttade']
Stem: maga, Suffixes: ['nannu']
Stem: maganan, Suffixes: ['nu']
Stem: magana, Suffixes: ['nnu']
Stem: oḍe, Suffixes: ['du']
Stem: saṃgīta, Suffixes: ['vannu']
Stem: saṃgītavan, Suffixes: ['nu']
Stem: saṃgītava, Suffixes: ['nnu']
Stem: kliṣṭa-v-ā, Suffixes: ['da']
Stem: ellari, Suffixes: ['giṃta']
Stem: kali, Suffixes: ['yalu']
Stem: hiṃ, Suffixes: ['de']
Stem: taṃ, Suffix

In [ ]:
suffix_info = {}

suffix_to_stems = defaultdict(list)  # suffix → [stems]
for stem, suffixes in stems_with_suffixes.items():
    for suffix in suffixes:
        suffix_to_stems[suffix].append(stem)

for suffix, stems in suffix_to_stems.items():
    # Create dict for this suffix
    suffix_info[suffix] = {
        "count": {},  # Will store stem:count pairs
    }
    
    # Count occurrences of each stem
    for stem in stems:
        if stem in suffix_info[suffix]["count"]:
            suffix_info[suffix]["count"][stem] += 1
        else:
            suffix_info[suffix]["count"][stem] = 1

# Display summary
print("Created info dictionary for", len(suffix_info), "suffixes")
suffix_info

Created info dictionary for 40 suffixes


{'vā': {'count': {'para': 1, 'pustaka': 1, 'mara': 1, 'atha': 1}},
 'ge': {'count': {'baruvahā': 1,
   'nana': 1,
   'ūri': 1,
   'keḷa': 1,
   'mane': 1,
   'avaḷi': 1,
   'hī': 1,
   'huḍugani': 1,
   'nima': 1,
   'avari': 1,
   'avani': 1,
   'hā': 1,
   'girākigaḷi': 1,
   'ēḷi': 1,
   'meravaṇi': 1,
   'kaḍe': 1,
   'manuṣyani': 1,
   'tama': 1,
   'nama': 1,
   'bage': 1,
   'maisūri': 1,
   'oḷa': 1,
   'ēḷ': 1,
   'māḍidahā': 1}},
 'nu': {'count': {'idan': 1,
   'hēḷida': 1,
   'viṣayavan': 1,
   'maganan': 1,
   'saṃgītavan': 1,
   'ellaran': 1,
   'patravan': 1,
   'ōduttiruve': 1,
   'nūran': 1,
   'tappan': 1,
   'biḍuve': 1,
   'baruttidde': 1,
   'nōḍuve': 1,
   'manuṣyanan': 1,
   'doḍḍava': 1,
   'nannan': 1,
   'malagide': 1,
   'haṇṇugaḷan': 1,
   'pustakagaḷan': 1,
   'koṭṭa': 1,
   'mūran': 1,
   'eraḍan': 1,
   'hūvan': 1,
   'tā': 1,
   'pustakavan': 1,
   'hōde': 1,
   'hōda': 1,
   'huḍuganan': 1,
   'citragaḷan': 1,
   'nīvē': 1,
   'kelasavan': 1,
   'ōduttid

In [ ]:
from collections import Counter

# Go through the results again, word by word, and check if the word has a stem with a suffix

for result in results:
    sentence, translated_sentence, pos_info = result
    transliterations = sentence.split()
    
    # Convert pos_info.items() to a list to support indexing
    pos_items = list(pos_info.items())
    
    for i,(word, info) in enumerate(pos_items):
        if info['affix'] is not None:
            prefix = info['affix']['prefix']
            suffix = info['affix']['suffix']
            
            # Check if the suffix is in the productive suffixes
            if suffix in productive_suffixes:
                # Add POS tag for the stem, POS tag for the previous word and POS tag for the next word into suffix_info[suffix]
                stem = word[:-len(suffix)] if len(suffix) > 0 else word
                
                # Get POS tags
                stem_pos = info.get('pos', [])
                prev_pos = pos_items[i-1][1].get('pos', []) if i > 0 else ['<BOS>']  # Beginning of sentence
                next_pos = pos_items[i+1][1].get('pos', []) if i < len(pos_items)-1 else ['<EOS>']  # End of sentence
                
                # Get translation tokens if available
                trans_tokens = []
                if translated_sentence:
                    trans_tokens = translated_sentence.lower().split()
                
                # Initialize or update the add_info stats for this suffix
                if suffix not in suffix_info:
                    suffix_info[suffix] = {
                        'count': defaultdict(int),
                        'stats': {
                            'stem_pos': Counter(),  # POS tags of stems
                            'pos_before': Counter(),  # POS tags of words before
                            'pos_after': Counter(),  # POS tags of words after
                            'trans_tokens': Counter(),  # Translation tokens
                            'lemma_set': set()  # Set of unique translations
                        }
                    }
                
                # Update statistics
                suffix_info[suffix]['count'][stem] += 1
                
                # Update POS tag counts
                if 'stats' not in suffix_info[suffix]:
                    suffix_info[suffix]['stats'] = {
                        'stem_pos': Counter(),
                        'pos_before': Counter(),
                        'pos_after': Counter(), 
                    }
                if stem_pos:
                    suffix_info[suffix]['stats']['stem_pos'].update(stem_pos)
                for pos in prev_pos:
                    suffix_info[suffix]['stats']['pos_before'][pos] += 1
                for pos in next_pos:
                    suffix_info[suffix]['stats']['pos_after'][pos] += 1

                # Update translation tokens
                if translated_sentence:  # Only add translation tokens if we have a translated sentence
                    if 'trans_tokens' not in suffix_info[suffix]['stats']:
                        suffix_info[suffix]['trans_tokens'] = []
                    try:
                        suffix_info[suffix]['stats']['trans_tokens'].update(trans_tokens)
                    except KeyError:
                        print(suffix_info[suffix]['stats'])
                # Add translation to lemma set if available
                if info.get('translation'):
                    suffix_info[suffix]['stats']['lemma_set'].add(info['translation'])

{'stem_pos': Counter({'NOUN': 2}), 'pos_before': Counter({'PRON': 4, 'PUNCT': 2}), 'pos_after': Counter({'<EOS>': 2})}


KeyError: 'lemma_set'

In [ ]:
# PRINT
for suffix, info in suffix_info.items():
    print(f"Suffix: {suffix}")
    print(f"  Count: {len(info['count'])} stems")
    #for stem, count in info['count'].items():
    #
    #     print(f"  Stem: {stem}, Count: {count}")
    if "add_info" in info:
        add_info = info["add_info"]
        print(f"  Stem: {add_info['stem']}, POS: {add_info['pos']}, Previous POS: {add_info['prev_pos']}, Next POS: {add_info['next_pos']}")
    print("=========================================")

Suffix: vā
  Count: 4 stems
Suffix: ge
  Count: 24 stems
Suffix: nu
  Count: 63 stems
Suffix: nnu
  Count: 29 stems
Suffix: gaḷu
  Count: 12 stems
Suffix: yā
  Count: 4 stems
Suffix: du
  Count: 28 stems
Suffix: lla
  Count: 34 stems
Suffix: de
  Count: 24 stems
Suffix: varu
  Count: 7 stems
Suffix: ttāne
  Count: 5 stems
Suffix: villa
  Count: 2 stems
Suffix: ru
  Count: 32 stems
Suffix: daru
  Count: 8 stems
Suffix: da
  Count: 25 stems
Suffix: ddēne
  Count: 10 stems
Suffix: vannu
  Count: 6 stems
Suffix: va
  Count: 6 stems
Suffix: vudilla
  Count: 6 stems
Suffix: ttēne
  Count: 8 stems
Suffix: venu
  Count: 7 stems
Suffix: ttade
  Count: 8 stems
Suffix: nannu
  Count: 5 stems
Suffix: giṃta
  Count: 3 stems
Suffix: yalu
  Count: 2 stems
Suffix: valla
  Count: 3 stems
Suffix: gaḷannu
  Count: 4 stems
Suffix: dāgi
  Count: 2 stems
Suffix: -
  Count: 6 stems
Suffix: ttiddenu
  Count: 2 stems
Suffix: dalli
  Count: 4 stems
Suffix: vāgali
  Count: 2 stems
Suffix: nige
  Count: 3 stems
S

# Unimorph morphology mapping

In [ ]:
import pandas as pd

df = pd.read_csv("data/unimorph_eng.txt",sep="\t",header=None)

df

,0,1,2
0,microtome,microtomes,N;PL
1,microtome,microtomes,V;PRS;3;SG
2,microtome,microtoming,V;V.PTCP;PRS
3,microtome,microtomed,V;PST
4,microtome,microtomed,V;V.PTCP;PST
...,...,...,...
652472,myriadaire,myriadaire,N;SG
652473,dibridgehead,dibridgehead,N;SG
652474,Chicagoese,Chicagoese,N;SG
652475,Druzer,Druzer,N;SG


In [ ]:
# For each row in the dictionary of kan to eng, check the mapping for the eng
import re
from tqdm import tqdm
count = 0

no_unimorph = []
unimorph_mapping = {}

for row in tqdm(kan_to_eng_mapping.items()):
    kan_word = row[0]
    eng_word = row[1]

    # clean the eng word
    
    # Remove content in brackets (Unless the entire content is in brackets)
    if eng_word.startswith('(') and eng_word.endswith(')'):
        eng_word_clean = eng_word.strip('()').strip()
    else:
        eng_word_clean = re.sub(r'\(.*?\)', '', eng_word).strip()
    # If there is a comma, split by it and take the first part TODO: Check if both words give same unimorph
    if ',' in eng_word_clean:
        eng_word_clean = eng_word_clean.split(',')[0].strip()

    # If there is a / in the eng_word, split by it and take the first part
    if '/' in eng_word_clean:
        eng_word_clean = eng_word_clean.split('/')[0].strip()

    if ';' in eng_word_clean:
        eng_word_clean = eng_word_clean.split(';')[0].strip()

    if eng_word_clean == "":
        print(eng_word)

    # Check if the eng_word is in the df
    if eng_word_clean in df[1].values:
        

        
        # Get the row index
        row_index = df[df[1] == eng_word_clean].index[0]
        
        # Get the unimorph value
        unimorph_value = df.iloc[row_index]
        unimorph_value_str = "|".join(unimorph_value)
        #print(f"{kan_word} -> {eng_word_clean}, Unimorph: {str(unimorph_value)}")
        unimorph_mapping[kan_word] = {
            "eng_word": eng_word_clean,
            "eng_word_full": eng_word,
            "unimorph": unimorph_value[2],
            "stem": unimorph_value[0],
        }
        count+=1
    else:
        #print(f"{kan_word} -> {eng_word}, Unimorph: Not found")
        if eng_word_clean.startswith("to"):

            # Just get the verb form and add infinitive form
            eng_word_verb = eng_word_clean.replace("to", "").strip()
            if eng_word_verb in df[1].values:
                row_index = df[df[1] == eng_word_verb].index[0]
                unimorph_value = df.iloc[row_index]
                unimorph_value_str = "|".join(unimorph_value)
                #print(f"{kan_word} -> {eng_word_clean}, Unimorph: {str(unimorph_value)}")
                unimorph_mapping[kan_word] = {
                    "eng_word": eng_word_clean,
                    "eng_word_full": eng_word,
                    "unimorph": "V;NFIN",
                    "unimorph_value_str": unimorph_value_str,
                    "stem": unimorph_value[0],
                }
                count+=1
            else:

                no_unimorph.append((eng_word_clean,eng_word))

print(f"Total number of words with unimorph mapping: {count} / {len(kan_to_eng_mapping)}")


100%|██████████| 773/773 [00:21<00:00, 35.24it/s]

Total number of words with unimorph mapping: 534 / 773


In [ ]:
no_unimorph

[('top side', 'top side'),
 ('to lie down', 'to lie down, sleep'),
 ('to set out', 'to set out, leave for'),
 ('to cause to call', 'to cause to call'),
 ('to be obtained', 'to be obtained, to be met'),
 ('to be seen', 'to be seen, appear'),
 ('today’s', 'today’s, of today'),
 ('together with', 'together with'),
 ('to everything', 'to everything (dative)'),
 ('to the tenth man', 'to the tenth man'),
 ('to sit down', 'to sit down'),
 ('to be defeated', 'to be defeated'),
 ('to sink down', 'to sink down / collapse'),
 ('to be tired', 'to be tired'),
 ('to be obtained', 'to be obtained'),
 ('to speak falsehood', 'to speak falsehood / lie'),
 ('to carry off', 'to carry off'),
 ('to become wet', 'to become wet'),
 ('to be hidden', 'to be hidden'),
 ('to become branched', 'to become branched'),
 ('to be', 'to be'),
 ('to be bored', 'to be bored'),
 ('to give birth', 'to give birth'),
 ('to bear a burden', 'to bear a burden'),
 ('to be spoilt', 'to be spoilt'),
 ('to put on', 'to put on (dress

In [ ]:
eng_word = "interest (neuter)"
eng_word = re.sub(r'\(.*?\)', '', eng_word).strip()
eng_word

'interest'

In [ ]:
unimorph_mapping

{'ōḍu': {'eng_word': 'to run',
  'eng_word_full': 'to run',
  'unimorph': 'V;V.PTCP;PST;NFIN',
  'unimorph_value_str': 'rin|run|V;V.PTCP;PST',
  'stem': 'rin'},
 'ōdu': {'eng_word': 'to read',
  'eng_word_full': 'to read',
  'unimorph': 'V;PST;NFIN',
  'unimorph_value_str': 'read|read|V;PST',
  'stem': 'read'},
 'hēḷu': {'eng_word': 'to say',
  'eng_word_full': 'to say',
  'unimorph': 'V;NFIN;IMP+SBJV;NFIN',
  'unimorph_value_str': 'say|say|V;NFIN;IMP+SBJV',
  'stem': 'say'},
 'hēlu': {'eng_word': 'to shit',
  'eng_word_full': 'to shit',
  'unimorph': 'V;PST;NFIN',
  'unimorph_value_str': 'shit|shit|V;PST',
  'stem': 'shit'},
 'baḍi': {'eng_word': 'to beat',
  'eng_word_full': 'to beat',
  'unimorph': 'V;PST;NFIN',
  'unimorph_value_str': 'beat|beat|V;PST',
  'stem': 'beat'},
 'baḍḍi': {'eng_word': 'interest',
  'eng_word_full': 'interest (on money)',
  'unimorph': 'N;SG',
  'stem': 'interest'},
 'nīnu': {'eng_word': 'you',
  'eng_word_full': 'you (singular and non-honorific)',
  'unim

In [ ]:
# Save unimorph mapping to a json file
with open("unimorph_mapping.json", "w", encoding="utf-8") as f:
    json.dump(unimorph_mapping, f, ensure_ascii=False, indent=4)

In [ ]:
# Go through the parallel sentences and create a data with the POS tag of each word and unimorph mapping of each word
parallel_sentences_with_pos_and_unimorph = []
for result in processed_sentences:
    sentence, translated_sentence, pos_info = result
    pos_unimorph_info = {}
    
    for word in sentence.split():
        word = word.lower()
        word = word.strip('.,!?()[]/"!')  # Remove punctuation from the word
        word = word.strip()  # Remove leading and trailing whitespace

        if word in pos_mapping:
            pos_unimorph_info[word] = {
                "pos": pos_mapping[word],
                "affix": None,  # No affix for this case
                "unimorph": unimorph_mapping.get(word, None)
            }
        elif word in kannada_pos_mapping:
            pos_unimorph_info[word] = {
                "pos": kannada_pos_mapping[word],
                "affix": None,  # No affix for this case
                "unimorph": unimorph_mapping.get(word, None)
            }
        else:
            # If the word is not found in any mapping, check the stemming dictionary
            found = False
            for affix, info in suffix_info.items():
                #if word.startswith("pustaka"):
                    #print(word,affix,word.endswith(affix),word.replace(affix,""))

                if word.endswith(affix):
                    word_stem = word.replace(affix,"") if len(affix) > 0 else word
                    if word_stem in pos_mapping:
                        pos_unimorph_info[word] = {
                            "pos": pos_mapping[word_stem],
                            "affix": affix,
                            "unimorph": unimorph_mapping.get(word_stem, None)
                        }
                        found = True
                        break
                    elif word_stem in kannada_pos_mapping:
                        pos_unimorph_info[word] = {
                            "pos": kannada_pos_mapping[word_stem],
                            "affix": affix,
                            "unimorph": unimorph_mapping.get(word_stem, None)
                        }
                        found = True
                        break
                    else:
                        # If the stem is not found, check if the affix is in the suffix_info
                        if affix in suffix_info:
                            pos_unimorph_info[word] = {
                                "pos": None,
                                "affix": affix,
                                "unimorph": None  # No unimorph mapping for this case
                            }
                

            # If still not found, add a placeholder


            if not found:
                pos_unimorph_info[word] = {
                    "pos": None,
                    "affix": None,  # No affix for this case
                    "unimorph": None
                }
        
    # Append the sentence with its translation and POS/unimorph info
    #print(f"Sentence: {sentence}")
    #print(f"Translated Sentence: {translated_sentence}")
    #print(f"POS/Unimorph Info: {pos_unimorph_info}")
    parallel_sentences_with_pos_and_unimorph.append({
        "sentence": sentence,
        "translated_sentence": translated_sentence,
        "pos_unimorph_info": pos_unimorph_info
    })

# Save the parallel sentences with POS and unimorph mapping to a json file


parallel_sentences_with_pos_and_unimorph

[{'sentence': 'Idu pustaka.',
  'translated_sentence': 'This is a book.',
  'pos_unimorph_info': {'idu': {'pos': ['PRON'],
    'affix': None,
    'unimorph': {'eng_word': 'it',
     'eng_word_full': 'it / this',
     'unimorph': 'V;PST',
     'stem': 'it'}},
   'pustaka': {'pos': ['NOUN'],
    'affix': None,
    'unimorph': {'eng_word': 'book',
     'eng_word_full': 'book',
     'unimorph': 'V;NFIN;IMP+SBJV',
     'stem': 'book'}}}},
 {'sentence': 'Ivu pustakagaḷu.',
  'translated_sentence': 'These are books.',
  'pos_unimorph_info': {'ivu': {'pos': ['PRON', 'PUNCT', 'PRON'],
    'affix': None,
    'unimorph': None},
   'pustakagaḷu': {'pos': ['NOUN'],
    'affix': 'gaḷu',
    'unimorph': {'eng_word': 'book',
     'eng_word_full': 'book',
     'unimorph': 'V;NFIN;IMP+SBJV',
     'stem': 'book'}}}},
 {'sentence': 'Adu mara.',
  'translated_sentence': 'That is a tree.',
  'pos_unimorph_info': {'adu': {'pos': ['PRON'],
    'affix': None,
    'unimorph': {'eng_word': 'it',
     'eng_word_f

In [39]:
# Save the parallel sentences with POS and unimorph mapping to a json file
with open("parallel_sentences_with_pos_and_unimorph.json", "w", encoding="utf-8") as f:
    json.dump(parallel_sentences_with_pos_and_unimorph, f, ensure_ascii=False, indent=4)

# Lemmatization and POS tagging

In [40]:
import spacy
import unimorph

nlp = spacy.load("en_core_web_sm")

sent = "I ate a cat"
doc  = nlp(sent)

for tok in doc:
    print(f"{tok.i:2}  {tok.text:<4}  "
          f"lemma={tok.lemma_:<5}  "
          f"upos={tok.pos_:<4}  "
          f"tag={tok.tag_:<4}  "
          f"morph={tok.morph}")


    
    


 0  I     lemma=I      upos=PRON  tag=PRP   morph=Case=Nom|Number=Sing|Person=1|PronType=Prs
 1  ate   lemma=eat    upos=VERB  tag=VBD   morph=Tense=Past|VerbForm=Fin
 2  a     lemma=a      upos=DET   tag=DT    morph=Definite=Ind|PronType=Art
 3  cat   lemma=cat    upos=NOUN  tag=NN    morph=Number=Sing


In [41]:
import json

with open("parallel_sentences_with_pos_and_unimorph.json","r", encoding="utf-8") as f:
    parallel_sentences_with_pos_and_unimorph = json.load(f)

In [42]:
len(parallel_sentences_with_pos_and_unimorph)

# Display the first 10 sentences with their POS and unimorph info
for i, sentence_data in enumerate(parallel_sentences_with_pos_and_unimorph[:10]):
    sentence = sentence_data['sentence']
    translated_sentence = sentence_data['translated_sentence']
    pos_unimorph_info = sentence_data['pos_unimorph_info']
    
    print(f"{i+1}. Sentence: {sentence}")
    print(f"   Translated Sentence: {translated_sentence}")

        

1. Sentence: Idu pustaka.
   Translated Sentence: This is a book.
2. Sentence: Ivu pustakagaḷu.
   Translated Sentence: These are books.
3. Sentence: Adu mara.
   Translated Sentence: That is a tree.
4. Sentence: Avu maragaḷu.
   Translated Sentence: Those are trees.
5. Sentence: Adu huḍuga.
   Translated Sentence: That is a boy.
6. Sentence: Adu mane.
   Translated Sentence: That is a house.
7. Sentence: Avu manegaḷu.
   Translated Sentence: Those are houses.
8. Sentence: Adu mahiḷe.
   Translated Sentence: That is a lady.
9. Sentence: Nānu vidyārthi.
   Translated Sentence: I am a [male] student.
10. Sentence: Nānu vidyārthini.
   Translated Sentence: I am a [female] student.


In [55]:
for sentence in parallel_sentences_with_pos_and_unimorph:
    translated_sentence = sentence["translated_sentence"]
    doc = nlp(translated_sentence)
    res = []
    tense = None
    genders = []
    number = None
    plural_details = None  # To store details if plural is detected
    genders_details = []
    for tok in doc:
        morph = tok.morph.to_dict()
        res.append({
            "word": tok.text,
            "lemma": tok.lemma_,
            "upos": tok.pos_,
            "tag": tok.tag_,
            "morph": morph,
        })

        if not number and morph.get("Number", None):
            number = morph.get("Number", None)
        elif morph.get("Number", None) == "Plur":
            number = "Plur"
            plural_details = {
                "word": tok.text, 
                "morph": morph
            }

        if not tense and morph.get("Tense", None):
            tense = morph.get("Tense", None)

        if morph.get("Gender", None):
            genders.append(morph.get("Gender", None))
            genders_details.append({
                "word": tok.text, 
                "morph": morph})

        
        
        #if morph.get("")


        
    #print(f"Sentence: {translated_sentence} \n Tense: {tense}, Number: {number}, Genders: {genders}")
    print(f"Sentence: {translated_sentence} \n Plural_details: {plural_details}, gender_details: {genders_details}")
    

        
    sentence["spacy_info"] = {"res": res, "tense": tense, "number": number, "plural_details": plural_details, "genders": genders,"genders_details":genders_details}

with open("parallel_sentences_with_additional_info.json", "w", encoding="utf-8") as f:
    json.dump(parallel_sentences_with_pos_and_unimorph, f, ensure_ascii=False, indent=4)

#parallel_sentences_with_pos_and_unimorph


Sentence: This is a book. 
 Plural_details: None, gender_details: []
Sentence: These are books. 
 Plural_details: {'word': 'books', 'morph': {'Number': 'Plur'}}, gender_details: []
Sentence: That is a tree. 
 Plural_details: None, gender_details: []
Sentence: Those are trees. 
 Plural_details: {'word': 'trees', 'morph': {'Number': 'Plur'}}, gender_details: []
Sentence: That is a boy. 
 Plural_details: None, gender_details: []
Sentence: That is a house. 
 Plural_details: None, gender_details: []
Sentence: Those are houses. 
 Plural_details: {'word': 'houses', 'morph': {'Number': 'Plur'}}, gender_details: []
Sentence: That is a lady. 
 Plural_details: None, gender_details: []
Sentence: I am a [male] student. 
 Plural_details: None, gender_details: []
Sentence: I am a [female] student. 
 Plural_details: None, gender_details: []
Sentence: this [is a] house [‘this thing – house’] 
 Plural_details: None, gender_details: []
Sentence: that [is a] building [‘that thing – building’] 
 Plural_det

In [57]:
parallel_sentences_with_pos_and_unimorph[0]["spacy_info"]

{'res': [{'word': 'This',
   'lemma': 'this',
   'upos': 'PRON',
   'tag': 'DT',
   'morph': {'Number': 'Sing', 'PronType': 'Dem'}},
  {'word': 'is',
   'lemma': 'be',
   'upos': 'AUX',
   'tag': 'VBZ',
   'morph': {'Mood': 'Ind',
    'Number': 'Sing',
    'Person': '3',
    'Tense': 'Pres',
    'VerbForm': 'Fin'}},
  {'word': 'a',
   'lemma': 'a',
   'upos': 'DET',
   'tag': 'DT',
   'morph': {'Definite': 'Ind', 'PronType': 'Art'}},
  {'word': 'book',
   'lemma': 'book',
   'upos': 'NOUN',
   'tag': 'NN',
   'morph': {'Number': 'Sing'}},
  {'word': '.',
   'lemma': '.',
   'upos': 'PUNCT',
   'tag': '.',
   'morph': {'PunctType': 'Peri'}}],
 'tense': 'Pres',
 'number': 'Sing',
 'plural_details': None,
 'genders': [],
 'genders_details': []}

# Unimorph analysis and inflection

In [83]:
result = unimorph.analyze_word("testing", lang="eng")

print(result)

test	testing	V;V.PTCP;PRS
testing	testing	ADJ
testing	testing	N;SG



In [87]:
result = unimorph.inflect_word("test",lang="eng")

print(result)

test	tests	N;PL
test	tests	V;PRS;3;SG
test	testing	V;V.PTCP;PRS
test	tested	V;PST
test	tested	V;V.PTCP;PST
test	test	N;SG
test	test	V;NFIN;IMP+SBJV



In [24]:
import json
# Load kalamang parallel sentences with POS and unimorph mapping
with open("parallel_sents/kalamang_parallel_sentences_with_pos_and_unimorph.json","r", encoding="utf-8") as f:
    kalamang_parallel_sentences = json.load(f)

In [25]:
for item in kalamang_parallel_sentences[:5]:
    print(item)

{'page': 52, 'IGT': '', 'spacy_info': {'res': [{'word': '[', 'lemma': '[', 'upos': 'X', 'tag': 'XX', 'morph': {}}, {'word': 'stim2_3:45', 'lemma': 'stim2_3:45', 'upos': 'NUM', 'tag': 'CD', 'morph': {'NumType': 'Card'}}, {'word': ']', 'lemma': ']', 'upos': 'PUNCT', 'tag': '-RRB-', 'morph': {'PunctSide': 'Fin', 'PunctType': 'Brck'}}, {'word': '‘', 'lemma': "'", 'upos': 'PUNCT', 'tag': '``', 'morph': {'PunctSide': 'Ini', 'PunctType': 'Quot'}}, {'word': 'The', 'lemma': 'the', 'upos': 'DET', 'tag': 'DT', 'morph': {'Definite': 'Def', 'PronType': 'Art'}}, {'word': 'dog', 'lemma': 'dog', 'upos': 'NOUN', 'tag': 'NN', 'morph': {'Number': 'Sing'}}, {'word': 'has', 'lemma': 'have', 'upos': 'AUX', 'tag': 'VBZ', 'morph': {'Mood': 'Ind', 'Number': 'Sing', 'Person': '3', 'Tense': 'Pres', 'VerbForm': 'Fin'}}, {'word': 'bitten', 'lemma': 'bite', 'upos': 'VERB', 'tag': 'VBN', 'morph': {'Aspect': 'Perf', 'Tense': 'Past', 'VerbForm': 'Part'}}, {'word': 'the', 'lemma': 'the', 'upos': 'DET', 'tag': 'DT', 'mo

In [26]:
import re



for row in kalamang_parallel_sentences:
    sentence = row["sentence"]
    translated_sentence = row["translated_sentence"]
    
    #print(f"Sentence: {sentence}")



    # Clean the translated sentence, remove any [] content at the start
    #translated_sentence_clean = re.sub(r'^\[.*?\]', '', translated_sentence).strip()
    # Take sentence to be inside first‘ and last’
    # last index of ’
    translated_sentence_clean = translated_sentence[translated_sentence.find('‘')+1:translated_sentence.rfind('’')]
    print(f"Cleaned Translated Sentence: {translated_sentence_clean}")
    print(f"--------Translated Sentence: {translated_sentence}")
    row["translated_sentence"] = translated_sentence_clean
    


Cleaned Translated Sentence: The dog has bitten the fish.
--------Translated Sentence: [stim2_3:45] ‘The dog has bitten the fish.’
Cleaned Translated Sentence: He sent me one hundred and fifty [thousand rupiah].
--------Translated Sentence: ‘He sent me one hundred and fifty [thousand rupiah].’ [conv12_3:09]
Cleaned Translated Sentence: They hugged the scared woman.
--------Translated Sentence: ‘They hugged the scared woman.’ [elic_adj19_8]
Cleaned Translated Sentence: I already wanted to sleep.
--------Translated Sentence: ‘I already wanted to sleep.’ [narr32_0:18]
Cleaned Translated Sentence: He said: “Hey, these fish, how [should we treat them]? Do we sell them, or do we split and salt them?”
--------Translated Sentence: ‘He said: “Hey, these fish, how [should we treat them]? Do we sell them, or do we split and salt them?”’ [narr8_5:34]
Cleaned Translated Sentence: He/she hits the small tree.
--------Translated Sentence: ‘He/she hits the small tree.’
Cleaned Translated Sentence: He a

In [27]:
# Save cleaned kalamang parallel sentences with POS and unimorph mapping
with open("parallel_sents/kalamang_parallel_sentences_with_pos_and_unimorph.json", "w", encoding="utf-8") as f:
    json.dump(kalamang_parallel_sentences, f, ensure_ascii=False, indent=4)